# Step 2 — Entity Construction
Membangun tabel Node dan Relationship yang siap diimport ke Neo4j.

```
OUTPUT (Neo4j-ready CSV)
├── nodes/
│   ├── nodes_movie.csv
│   ├── nodes_actor.csv
│   ├── nodes_director.csv
│   ├── nodes_genre.csv
│   └── nodes_keyword.csv
└── relationships/
    ├── rels_movie_actor.csv
    ├── rels_movie_director.csv
    ├── rels_movie_genre.csv
    └── rels_movie_keyword.csv
```

In [1]:
import pandas as pd
import ast
import os

DATASET_DIR = os.path.join('..', 'dataset')
NODE_DIR    = os.path.join('..', 'entity_construction', 'nodes')
REL_DIR     = os.path.join('..', 'entity_construction', 'relationships')

os.makedirs(NODE_DIR, exist_ok=True)
os.makedirs(REL_DIR,  exist_ok=True)

def safe_parse(val):
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return []

## Load Data Hasil Parsing

In [2]:
df_movies   = pd.read_csv(os.path.join(DATASET_DIR, 'movies_clean.csv'))
df_credits  = pd.read_csv(os.path.join(DATASET_DIR, 'credits_clean.csv'))
df_keywords = pd.read_csv(os.path.join(DATASET_DIR, 'keywords_clean.csv'))

# Kolom genres/actors/directors/keywords masih berupa string list → parse lagi
df_movies['genres']       = df_movies['genres'].apply(safe_parse)
df_credits['actors']      = df_credits['actors'].apply(safe_parse)
df_credits['directors']   = df_credits['directors'].apply(safe_parse)
df_keywords['keywords']   = df_keywords['keywords'].apply(safe_parse)

print('movies  :', df_movies.shape)
print('credits :', df_credits.shape)
print('keywords:', df_keywords.shape)

movies  : (45463, 5)
credits : (45476, 3)
keywords: (46419, 2)


---
## NODE 1 — Movie
Kolom: `movie_id`, `title`, `release_date`, `revenue`

In [3]:
nodes_movie = df_movies[['id', 'title', 'release_date', 'revenue']].copy()
nodes_movie = nodes_movie.rename(columns={'id': 'movie_id'})
nodes_movie = nodes_movie.dropna(subset=['movie_id', 'title'])
nodes_movie['movie_id'] = nodes_movie['movie_id'].astype(int)

print(f'Total Movie nodes: {len(nodes_movie)}')
nodes_movie.head(3)

Total Movie nodes: 45460


,movie_id,title,release_date,revenue
0,862,Toy Story,1995-10-30,373554033.0
1,8844,Jumanji,1995-12-15,262797249.0
2,15602,Grumpier Old Men,1995-12-22,0.0


In [4]:
nodes_movie.to_csv(os.path.join(NODE_DIR, 'nodes_movie.csv'), index=False)
print('Tersimpan: nodes/nodes_movie.csv')

Tersimpan: nodes/nodes_movie.csv


---
## NODE 2 — Actor
Kolom: `actor_id`, `name`

In [5]:
# Kumpulkan semua nama aktor unik dari seluruh film
all_actors = set()
for actors_list in df_credits['actors']:
    all_actors.update(actors_list)

nodes_actor = pd.DataFrame(sorted(all_actors), columns=['name'])
nodes_actor.insert(0, 'actor_id', range(1, len(nodes_actor) + 1))

print(f'Total Actor nodes: {len(nodes_actor)}')
nodes_actor.head(3)

Total Actor nodes: 73155


,actor_id,name
0,1,\tRobert Osth
1,2,Alistair Freeland
2,3,Belen Blanco


In [6]:
nodes_actor.to_csv(os.path.join(NODE_DIR, 'nodes_actor.csv'), index=False)
print('Tersimpan: nodes/nodes_actor.csv')

Tersimpan: nodes/nodes_actor.csv


---
## NODE 3 — Director
Kolom: `director_id`, `name`

In [7]:
all_directors = set()
for dir_list in df_credits['directors']:
    all_directors.update(dir_list)

nodes_director = pd.DataFrame(sorted(all_directors), columns=['name'])
nodes_director.insert(0, 'director_id', range(1, len(nodes_director) + 1))

print(f'Total Director nodes: {len(nodes_director)}')
nodes_director.head(3)

Total Director nodes: 19740


,director_id,name
0,1,Dale Trevillion\t
1,2,Davide Manuli
2,3,E.W. Swackhamer


In [8]:
nodes_director.to_csv(os.path.join(NODE_DIR, 'nodes_director.csv'), index=False)
print('Tersimpan: nodes/nodes_director.csv')

Tersimpan: nodes/nodes_director.csv


---
## NODE 4 — Genre
Kolom: `genre_id`, `name`

In [9]:
all_genres = set()
for genre_list in df_movies['genres']:
    all_genres.update(genre_list)

nodes_genre = pd.DataFrame(sorted(all_genres), columns=['name'])
nodes_genre.insert(0, 'genre_id', range(1, len(nodes_genre) + 1))

print(f'Total Genre nodes: {len(nodes_genre)}')
nodes_genre.head(5)

Total Genre nodes: 20


,genre_id,name
0,1,Action
1,2,Adventure
2,3,Animation
3,4,Comedy
4,5,Crime


In [10]:
nodes_genre.to_csv(os.path.join(NODE_DIR, 'nodes_genre.csv'), index=False)
print('Tersimpan: nodes/nodes_genre.csv')

Tersimpan: nodes/nodes_genre.csv


---
## NODE 5 — Keyword
Kolom: `keyword_id`, `name`

In [11]:
all_keywords = set()
for kw_list in df_keywords['keywords']:
    all_keywords.update(kw_list)

nodes_keyword = pd.DataFrame(sorted(all_keywords), columns=['name'])
nodes_keyword.insert(0, 'keyword_id', range(1, len(nodes_keyword) + 1))

print(f'Total Keyword nodes: {len(nodes_keyword)}')
nodes_keyword.head(3)

Total Keyword nodes: 19956


,keyword_id,name
0,1,'comfort women'
1,2,077
2,3,10th century


In [12]:
nodes_keyword.to_csv(os.path.join(NODE_DIR, 'nodes_keyword.csv'), index=False)
print('Tersimpan: nodes/nodes_keyword.csv')

Tersimpan: nodes/nodes_keyword.csv


---
## RELATIONSHIPS
Setiap tabel berisi pasangan `movie_id` ↔ entitas untuk membentuk edge di graph.

In [13]:
# Gabungkan movies + credits berdasarkan id
df_movies['id'] = df_movies['id'].astype(int)
df_credits['id'] = df_credits['id'].astype(int)
df_keywords['id'] = df_keywords['id'].astype(int)

df_merged = df_movies.merge(df_credits,  on='id', how='inner')
df_merged = df_merged.merge(df_keywords, on='id', how='inner')
print(f'Total film setelah merge: {len(df_merged)}')

Total film setelah merge: 46628


### REL 1 — Movie → actedBy → Actor

In [14]:
rels_movie_actor = df_merged[['id', 'actors']].explode('actors').dropna()
rels_movie_actor = rels_movie_actor.rename(columns={'id': 'movie_id', 'actors': 'actor_name'})
rels_movie_actor = rels_movie_actor[rels_movie_actor['actor_name'].str.strip() != '']

print(f'Total relasi movie-actor: {len(rels_movie_actor)}')
rels_movie_actor.head(3)

Total relasi movie-actor: 207535


,movie_id,actor_name
0,862,Tom Hanks
0,862,Tim Allen
0,862,Don Rickles


In [15]:
rels_movie_actor.to_csv(os.path.join(REL_DIR, 'rels_movie_actor.csv'), index=False)
print('Tersimpan: relationships/rels_movie_actor.csv')

Tersimpan: relationships/rels_movie_actor.csv


### REL 2 — Movie → directedBy → Director

In [16]:
rels_movie_director = df_merged[['id', 'directors']].explode('directors').dropna()
rels_movie_director = rels_movie_director.rename(columns={'id': 'movie_id', 'directors': 'director_name'})
rels_movie_director = rels_movie_director[rels_movie_director['director_name'].str.strip() != '']

print(f'Total relasi movie-director: {len(rels_movie_director)}')
rels_movie_director.head(3)

Total relasi movie-director: 50295


,movie_id,director_name
0,862,John Lasseter
1,8844,Joe Johnston
2,15602,Howard Deutch


In [17]:
rels_movie_director.to_csv(os.path.join(REL_DIR, 'rels_movie_director.csv'), index=False)
print('Tersimpan: relationships/rels_movie_director.csv')

Tersimpan: relationships/rels_movie_director.csv


### REL 3 — Movie → hasGenre → Genre

In [18]:
rels_movie_genre = df_merged[['id', 'genres']].explode('genres').dropna()
rels_movie_genre = rels_movie_genre.rename(columns={'id': 'movie_id', 'genres': 'genre_name'})
rels_movie_genre = rels_movie_genre[rels_movie_genre['genre_name'].str.strip() != '']

print(f'Total relasi movie-genre: {len(rels_movie_genre)}')
rels_movie_genre.head(3)

Total relasi movie-genre: 93342


,movie_id,genre_name
0,862,Animation
0,862,Comedy
0,862,Family


In [19]:
rels_movie_genre.to_csv(os.path.join(REL_DIR, 'rels_movie_genre.csv'), index=False)
print('Tersimpan: relationships/rels_movie_genre.csv')

Tersimpan: relationships/rels_movie_genre.csv


### REL 4 — Movie → hasKeyword → Keyword

In [20]:
rels_movie_keyword = df_merged[['id', 'keywords']].explode('keywords').dropna()
rels_movie_keyword = rels_movie_keyword.rename(columns={'id': 'movie_id', 'keywords': 'keyword_name'})
rels_movie_keyword = rels_movie_keyword[rels_movie_keyword['keyword_name'].str.strip() != '']

print(f'Total relasi movie-keyword: {len(rels_movie_keyword)}')
rels_movie_keyword.head(3)

Total relasi movie-keyword: 159437


,movie_id,keyword_name
0,862,jealousy
0,862,toy
0,862,boy


In [21]:
rels_movie_keyword.to_csv(os.path.join(REL_DIR, 'rels_movie_keyword.csv'), index=False)
print('Tersimpan: relationships/rels_movie_keyword.csv')

Tersimpan: relationships/rels_movie_keyword.csv


---
## Ringkasan Akhir

In [22]:
print('=== NODE SUMMARY ===')
for name, df in [
    ('nodes_movie',    nodes_movie),
    ('nodes_actor',    nodes_actor),
    ('nodes_director', nodes_director),
    ('nodes_genre',    nodes_genre),
    ('nodes_keyword',  nodes_keyword),
]:
    print(f'  {name:<20} : {len(df):>6} nodes')

print('\n=== RELATIONSHIP SUMMARY ===')
for name, df in [
    ('rels_movie_actor',    rels_movie_actor),
    ('rels_movie_director', rels_movie_director),
    ('rels_movie_genre',    rels_movie_genre),
    ('rels_movie_keyword',  rels_movie_keyword),
]:
    print(f'  {name:<25} : {len(df):>7} edges')

=== NODE SUMMARY ===
  nodes_movie          :  45460 nodes
  nodes_actor          :  73155 nodes
  nodes_director       :  19740 nodes
  nodes_genre          :     20 nodes
  nodes_keyword        :  19956 nodes

=== RELATIONSHIP SUMMARY ===
  rels_movie_actor          :  207535 edges
  rels_movie_director       :   50295 edges
  rels_movie_genre          :   93342 edges
  rels_movie_keyword        :  159437 edges
